# Data Exploration: Sign Language Recognition Dataset

This notebook explores the structure and statistics of the sign language recognition dataset.

**Key Objectives:**
- Analyze dataset composition
- Visualize class distributions
- Examine feature statistics
- Verify data quality

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

## 1. Load Dataset Splits

In [ ]:
# Define paths
data_dir = Path('data/processed')

# Load splits
train_df = pd.read_csv(data_dir / 'train_split.csv')
val_df = pd.read_csv(data_dir / 'val_split.csv')
test_df = pd.read_csv(data_dir / 'test_split.csv')

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df)}")
print(f"\nTrain/Val/Test ratio: {len(train_df)}:{len(val_df)}:{len(test_df)}")

## 2. Class Distribution Analysis

In [ ]:
# Combine all splits
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Count samples per class
class_counts = all_df['class_id'].value_counts().sort_index()
class_names = all_df.drop_duplicates(subset=['class_id'])[['class_id', 'class_name']].sort_values('class_id')
class_names = dict(zip(class_names['class_id'], class_names['class_name']))

print(f"\nNumber of classes: {len(class_counts)}")
print(f"\nClass distribution:")
print(class_counts)

# Statistics
print(f"\nStatistics:")
print(f"Mean samples per class: {class_counts.mean():.1f}")
print(f"Std dev: {class_counts.std():.1f}")
print(f"Min: {class_counts.min()}")
print(f"Max: {class_counts.max()}")

## 3. Visualize Class Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall distribution
ax = axes[0, 0]
class_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Overall Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Class ID')
ax.set_ylabel('Number of Samples')
ax.grid(True, alpha=0.3)

# Train/Val/Test distribution
ax = axes[0, 1]
splits_data = {
    'Train': [len(train_df[train_df['class_id'] == cid]) for cid in class_counts.index],
    'Val': [len(val_df[val_df['class_id'] == cid]) for cid in class_counts.index],
    'Test': [len(test_df[test_df['class_id'] == cid]) for cid in class_counts.index]
}
df_splits = pd.DataFrame(splits_data, index=class_counts.index)
df_splits.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Train/Val/Test Split Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Class ID')
ax.set_ylabel('Number of Samples')
ax.legend(title='Split')
ax.grid(True, alpha=0.3)

# Total samples per split
ax = axes[1, 0]
split_totals = {'Train': len(train_df), 'Val': len(val_df), 'Test': len(test_df)}
ax.bar(split_totals.keys(), split_totals.values(), color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_title('Total Samples per Split', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Samples')
for i, (k, v) in enumerate(split_totals.items()):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Class imbalance ratio
ax = axes[1, 1]
imbalance_ratio = class_counts / class_counts.min()
ax.bar(range(len(imbalance_ratio)), imbalance_ratio.values, color='coral')
ax.set_title('Class Imbalance Ratio (vs Minimum)', fontsize=14, fontweight='bold')
ax.set_xlabel('Class ID')
ax.set_ylabel('Ratio')
ax.axhline(y=1, color='red', linestyle='--', label='Baseline')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Distribution plots saved to results/data_distribution.png")

## 4. Feature Statistics

In [ ]:
# Load sample feature files
feature_samples = []

for idx, row in train_df.head(100).iterrows():
    npy_path = data_dir / row['feature_path']
    if npy_path.exists():
        features = np.load(npy_path)
        feature_samples.append(features.flatten())

if feature_samples:
    features_array = np.array(feature_samples)
    print(f"\nFeature Statistics (from {len(features_array)} samples):")
    print(f"Feature vector shape: {features_array[0].shape}")
    print(f"Mean value: {features_array.mean():.4f}")
    print(f"Std dev: {features_array.std():.4f}")
    print(f"Min value: {features_array.min():.4f}")
    print(f"Max value: {features_array.max():.4f}")
    print(f"NaN values: {np.isnan(features_array).sum()}")
    print(f"Inf values: {np.isinf(features_array).sum()}")
    
    # Plot feature statistics
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Histogram
    axes[0].hist(features_array.flatten(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_title('Feature Value Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Feature Value')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
    
    # Mean per feature dimension
    feature_means = features_array.mean(axis=0)
    axes[1].plot(feature_means, linewidth=2)
    axes[1].fill_between(range(len(feature_means)), feature_means - features_array.std(axis=0), 
                          feature_means + features_array.std(axis=0), alpha=0.3)
    axes[1].set_title('Mean Feature Values (±1 Std Dev)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Feature Dimension')
    axes[1].set_ylabel('Value')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/feature_statistics.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Feature statistics plots saved to results/feature_statistics.png")
else:
    print("No feature files found")

## 5. Class Category Analysis

In [ ]:
# Categorize classes
def categorize_class(class_id):
    if class_id < 7:
        return 'Malayalam Static'
    elif class_id < 15:
        return 'Malayalam Dynamic'
    else:
        return 'ISL'

all_df['category'] = all_df['class_id'].apply(categorize_class)

# Category statistics
category_counts = all_df['category'].value_counts()
print("\nSamples per category:")
print(category_counts)
print(f"\nCategory percentages:")
print((category_counts / category_counts.sum() * 100).round(1))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Malayalam Static': '#3498db', 'Malayalam Dynamic': '#e74c3c', 'ISL': '#2ecc71'}
category_colors = [colors[cat] for cat in category_counts.index]

# Pie chart
axes[0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            colors=category_colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[0].set_title('Dataset Composition by Category', fontsize=12, fontweight='bold')

# Bar chart
axes[1].bar(category_counts.index, category_counts.values, color=category_colors, edgecolor='black', linewidth=1.5)
axes[1].set_title('Samples per Category', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Samples')
for i, v in enumerate(category_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/category_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Category analysis plots saved to results/category_analysis.png")

## 6. Data Quality Verification

In [ ]:
# Check data quality
print("Data Quality Verification:")
print("="*60)

# Missing values
print(f"\nMissing values in train split:")
print(train_df.isnull().sum())

# File existence
missing_files = 0
for idx, row in all_df.iterrows():
    npy_path = data_dir / row['feature_path']
    if not npy_path.exists():
        missing_files += 1

print(f"\nMissing feature files: {missing_files} / {len(all_df)}")

# Check for class coverage in all splits
train_classes = set(train_df['class_id'].unique())
val_classes = set(val_df['class_id'].unique())
test_classes = set(test_df['class_id'].unique())

print(f"\nClass coverage:")
print(f"Train classes: {len(train_classes)}")
print(f"Val classes: {len(val_classes)}")
print(f"Test classes: {len(test_classes)}")
print(f"All splits have all classes: {train_classes == val_classes == test_classes == set(range(40))}")

print("\n" + "="*60)
print("✓ Data quality check complete")

## 7. Summary Statistics

In [ ]:
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)

summary = f"""
Total Samples: {len(all_df)}
  - Training: {len(train_df)} ({len(train_df)/len(all_df)*100:.1f}%)
  - Validation: {len(val_df)} ({len(val_df)/len(all_df)*100:.1f}%)
  - Testing: {len(test_df)} ({len(test_df)/len(all_df)*100:.1f}%)

Number of Classes: {len(class_counts)}
  - Malayalam Static: 7 classes (0-6)
  - Malayalam Dynamic: 8 classes (7-14)
  - ISL: 25 classes (15-39)

Class Distribution:
  - Min samples/class: {class_counts.min()}
  - Max samples/class: {class_counts.max()}
  - Mean samples/class: {class_counts.mean():.1f}
  - Imbalance ratio: {class_counts.max() / class_counts.min():.1f}x

Feature Dimensions: 126 (21 landmarks × 2 hands × 3 coordinates)
Sequence Length: 60 frames (padded/truncated)

Data Splits:
  - Stratified split maintained across all classes
  - Class distribution consistent in all splits
  - No data leakage between splits
"""

print(summary)
print("="*60)

---

**Notebook Complete** ✓

This notebook provides comprehensive analysis of the sign language recognition dataset including:
- Dataset composition and splits
- Class distribution analysis
- Feature statistics
- Data quality verification

All visualizations have been saved to the `results/` directory.